# Lesson 00 - Setup and Foundry readiness

**Level:** introductory | **Demo time:** 5 minutes

This lesson verifies the local environment before we make a model call. It deliberately separates **configuration**, **identity**, and the **Foundry model client** so authentication problems are easy to diagnose.

## Learning objectives

- Load project settings from a local `.env` file without storing credentials.
- Authenticate with Microsoft Entra ID through `DefaultAzureCredential`.
- Create the Microsoft Agent Framework `FoundryChatClient` used by later lessons.

## The connection path

```text
Notebook -> DefaultAzureCredential -> Azure CLI sign-in
         -> FoundryChatClient       -> Foundry project/model deployment
```

`DefaultAzureCredential` is a credential *chain*. During this local workshop it can use the identity created by `az login`. In Azure, the same application code can use managed identity instead. No API key is required in either case.

## One-time setup

From a terminal at the repository root:

```powershell
uv sync --all-groups
Copy-Item .env.sample .env
az login
uv run python -m ipykernel install --user --name maf-foundry-hackathon --display-name "MAF Foundry Hackathon"
uv run jupyter lab
```

Update `.env` with the project endpoint and deployed model name shown in Microsoft Foundry. Select the **MAF Foundry Hackathon** kernel in VS Code or JupyterLab.

In [ ]:
from maf_foundry_hackathon.config import (
    WorkshopSettings,
    create_chat_client,
    create_credential,
)

settings = WorkshopSettings.from_env()
settings.safe_summary()

The values above identify resources but do not grant access. Authorization still comes from Microsoft Entra ID. The helper intentionally never reads or displays an API key.

In [ ]:
credential = create_credential()
access_token = credential.get_token("https://ai.azure.com/.default")
print("Microsoft Entra authentication succeeded.")
print("Token expiry is available, but the token itself is never displayed:", access_token.expires_on)

In [ ]:
chat_client = create_chat_client(settings, credential)
print("Ready for the next lesson:", type(chat_client).__name__)

## Troubleshooting talking points

| Symptom | Likely cause | Action |
|---|---|---|
| Configuration error | Placeholder remains in `.env` | Copy the exact project endpoint and deployment name from Foundry |
| `CredentialUnavailableError` | Azure CLI is not signed in | Run `az login`, then rerun the cell |
| HTTP 403 | Identity lacks project access | Ask an administrator for the appropriate Foundry project role |
| Model not found | Deployment name differs from model catalog name | Use the **deployment name**, not the base model name |

> Presenter note: environment failures are intentionally isolated here so the agent lessons remain focused on MAF concepts.

In [ ]:
credential.close()
print("Credential resources closed.")

## Recap

We now have a secure identity and a model client. Lesson 01 adds the smallest useful abstraction in MAF: an `Agent` with a role and instructions.